## Cleaning

### Data Import

In [1]:
import polars as pl
import altair as alt
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # override=True ensures .env values take precedence over system env vars

USER = os.getenv("USER")
PASSWORD = os.getenv("PW")
HOST = os.getenv("HOST")
DATABASE = os.getenv("DB_NAME")

uri = f"postgresql://{USER}:{PASSWORD}@{HOST}:5432/{DATABASE}"

query = "SELECT * FROM daan_822.article_detailed"

df = pl.read_database_uri(query=query, uri=uri)

In [ ]:
df.describe()

### Cleaning
1. Whitespace in strings — strips leading/trailing whitespace from all string columns
  2. Empty strings instead of nulls — replaces "" with None for proper null semantics
  3. Wrong types for numeric columns — copyright_year, figure_count, table_count stored as Text in DB, cast to Int16
  4. pub_date stored as string — converts to proper Date type with multi-format fallback (YYYY-MM-DD, YYYY-MM, YYYY)
  5. license_type entirely null — drops the column
  6. Case inconsistency — lowercases keywords, journal_title, publisher_name
  7. JSON-as-string fields — parses authors and keywords from JSON strings into structured columns
  8. Remaining JSON array columns — detects them dynamically and converts to comma-separated strings

In [2]:
# cleaning whitespace and replacing blank with true nulls
df_cleaning = df.with_columns(pl.selectors.string().str.strip_chars().replace("", None))

# converting string type to int
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("copyright_year", "figure_count", "table_count").cast(pl.Int16)
)

# converting pub_date to date
df_cleaning = df_cleaning.with_columns(
    pl.coalesce(
        [
            pl.col("pub_date").str.to_date("%Y-%m-%d", strict=False),
            pl.col("pub_date").str.to_date("%Y-%m", strict=False),
            pl.col("pub_date").str.to_date("%Y", strict=False),
        ]
    )
)

# dropping column as it is null
df_cleaning = df_cleaning.drop("license_type")

# making columns lower case
df_cleaning = df_cleaning.with_columns(pl.selectors.by_name("keywords", "journal_title", "publisher_name").str.to_lowercase())

In [ ]:
df_cleaning.describe()

In [7]:
# labelling field names in the json construct for authors
dtypes = pl.List(
    pl.Struct(
        [
            pl.Field("given_names", pl.String),
            pl.Field("is_corresponding", pl.Boolean),
            pl.Field("orcid", pl.String),
            pl.Field("surname", pl.String),
        ]
    )
)

# parse JSON columns without exploding to avoid author x keyword cartesian product
df_parsed = df_cleaning.with_columns(
    pl.col("authors").str.json_decode(dtypes).alias("authors_struct"),
    pl.col("keywords").str.json_decode(pl.List(pl.String)).alias("keywords_list"),
).drop("authors", "keywords")

# identify columns that are json formatted in order to clean to be comma separated
json_array_cols = [
    col
    for col, dtype in df_parsed.schema.items()
    if dtype == pl.String
    and df_parsed.select(pl.col(col).drop_nulls().str.starts_with("[").all()).item()
]

# clean up JSON array columns to comma-separated strings
df_base = df_parsed.with_columns(
    [
        pl.col(col).str.json_decode(pl.List(pl.String)).list.join(", ").alias(col)
        for col in json_array_cols
    ]
)

# separate exploded views to avoid author x keyword cartesian product
df_authors = (
    df_base.explode("authors_struct")
    .unnest("authors_struct")
    .with_columns(
        (pl.col("given_names") + " " + pl.col("surname")).alias("author_full_name")
    )
)

df_keywords = df_base.explode("keywords_list").rename({"keywords_list": "keyword"})

pmid,doi,article_type,article_title,article_subject,pub_date,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available,authors_struct,keyword
str,str,str,str,str,date,i32,str,str,str,i16,str,str,bool,i16,i16,str,str,bool,str,bool,list[struct[4]],str
"""41215574""","""10.1097/PTS.0000000000001434""","""research-article""","""Analysis of Patient Safety Eve…","""The Health Care Manager Articl…",2025-11-11,31,"""journal of patient safety""","""springer international publish…","""Copyright © 2025 The Author(s)…",2026,"""Background:Diagnostic errors a…",""" *MedStar Health Research Inst…",false,4,0,"""""","""""",false,"""""",false,"[{""Patricia"",false,"""",""Spaar""}, {""Seth M."",false,"""",""Krevat""}, … {""Jeffrey A."",false,"""",""Gold""}]","""electronic health records"""
"""41215574""","""10.1097/PTS.0000000000001434""","""research-article""","""Analysis of Patient Safety Eve…","""The Health Care Manager Articl…",2025-11-11,31,"""journal of patient safety""","""springer international publish…","""Copyright © 2025 The Author(s)…",2026,"""Background:Diagnostic errors a…",""" *MedStar Health Research Inst…",false,4,0,"""""","""""",false,"""""",false,"[{""Patricia"",false,"""",""Spaar""}, {""Seth M."",false,"""",""Krevat""}, … {""Jeffrey A."",false,"""",""Gold""}]","""diagnostic error"""
"""41215574""","""10.1097/PTS.0000000000001434""","""research-article""","""Analysis of Patient Safety Eve…","""The Health Care Manager Articl…",2025-11-11,31,"""journal of patient safety""","""springer international publish…","""Copyright © 2025 The Author(s)…",2026,"""Background:Diagnostic errors a…",""" *MedStar Health Research Inst…",false,4,0,"""""","""""",false,"""""",false,"[{""Patricia"",false,"""",""Spaar""}, {""Seth M."",false,"""",""Krevat""}, … {""Jeffrey A."",false,"""",""Gold""}]","""patient safety event reports"""
"""41249591""","""10.1038/s41372-025-02472-1""","""research-article""","""Nasal intermittent positive pr…","""Article""",2025-11-17,28,"""journal of perinatology""",null,"""© The Author(s) 2025""",2025,"""ObjectiveWe describe a novel s…","""1https://ror.org/0011qv509grid…",true,4,2,"""""","""De-identified data is availabl…",true,"""""",false,"[{""Mark F."",false,""http://orcid.org/0000-0003-3518-8376"",""Weems""}, {""Vineet"",false,"""",""Lamba""}, … {""Rangasamy"",false,"""",""Ramanathan""}]","""respiratory tract diseases"""
"""41249591""","""10.1038/s41372-025-02472-1""","""research-article""","""Nasal intermittent positive pr…","""Article""",2025-11-17,28,"""journal of perinatology""",null,"""© The Author(s) 2025""",2025,"""ObjectiveWe describe a novel s…","""1https://ror.org/0011qv509grid…",true,4,2,"""""","""De-identified data is availabl…",true,"""""",false,"[{""Mark F."",false,""http://orcid.org/0000-0003-3518-8376"",""Weems""}, {""Vineet"",false,"""",""Lamba""}, … {""Rangasamy"",false,"""",""Ramanathan""}]","""paediatrics"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""40326708""","""10.1021/acssynbio.4c00807""","""research-article""","""Genetically Encoded Fluorescen…",null,2025-05-06,0,"""acs synthetic biology""",null,"""© 2025 The Authors. Published …",2025,"""Genetically encoded, single-ce…","""†Department of Chemical and Bi…",true,0,0,"""U.S. Department of Health and …","""""",false,"""""",false,"[{""Xiaoming"",false,"""",""Lu""}, {""Daniel J."",false,"""",""Pritko""}, … {""Marc R."",false,""https://orcid.org/0000-0002-0341-0705"",""Birtwistle""}]","""genetically-encoded fluorescen…"
"""40326708""","""10.1021/acssynbio.4c00807""","""research-article""","""Genetically Encoded Fluorescen…",null,2025-05-06,0,"""acs synthetic biology""",null,"""© 2025 The Authors. Published …",2025,"""Genetically encoded, single-ce…","""†Department of Chemical and Bi…",true,0,0,"""U.S. Department of Health and …","""""",false,"""""",false,"[{""Xiaoming"",false

In [ ]:
df_base.describe()

In [9]:
total_rows = df_base.height

cols_with_null_info = [
    (col, df_base.select(pl.col(col).null_count()).item(), round(df_base.select(pl.col(col).null_count()).item() / total_rows * 100, 2), df_base.schema[col])
    for col in df_base.columns
    if df_base.select(pl.col(col).null_count()).item() > 0
]

cols_with_null_info

[('doi', 11, 0.15, String),
 ('article_subject', 38, 0.53, String),
 ('pub_date', 22, 0.31, Date),
 ('publisher_name', 660, 9.19, String),
 ('copyright_statement', 307, 4.28, String),
 ('copyright_year', 1073, 14.95, Int16),
 ('abstract', 61, 0.85, String)]

In [10]:
# how many rows are missing copyright_year but have a pub_date?
missing_cy = df_base.filter(pl.col("copyright_year").is_null())

print(f"Missing copyright_year: {missing_cy.height}")
print(f"  - with pub_date available: {missing_cy.filter(pl.col('pub_date').is_not_null()).height}")
print(f"  - without pub_date:        {missing_cy.filter(pl.col('pub_date').is_null()).height}")

Missing copyright_year: 1073
  - with pub_date available: 1067
  - without pub_date:        6


In [ ]:
df_base = df_base.with_columns(
    pl.when(pl.col("copyright_year").is_null())
    .then(pl.col("pub_date").dt.year().cast(pl.Int16))
    .otherwise(pl.col("copyright_year"))
    .alias("copyright_year")
)

## Analysis

In [ ]:
df_base.schema

In [ ]:
df.describe()

In [ ]:
# creating a summary dataframe of common metrics
summary_df = pl.DataFrame(
    {
        "Metric": [
            "Total Articles",
            "Unique Authors",
            "Unique Keywords",
            "Avg Authors per Paper",
            "Avg Keywords per Paper",
        ],
        "Value": [
            df_base.select(pl.col("pmid").n_unique()).item(),
            df_authors.select(pl.col("author_full_name").n_unique()).item(),
            df_keywords.select(pl.col("keyword").n_unique()).item(),
            df_authors.group_by("pmid")
            .agg(pl.col("author_full_name").n_unique().alias("n_authors"))
            .select(pl.col("n_authors").mean().round(2))
            .item(),
            df_keywords.group_by("pmid")
            .agg(pl.col("keyword").n_unique().alias("n_keywords"))
            .select(pl.col("n_keywords").mean().round(2))
            .item(),
        ],
    },
    strict=False,
)

summary_df

#### Building Visuals

In [ ]:
data_avail = df_base.group_by("data_available").agg(pl.len().alias("count"))

data_avail

In [ ]:
pub_year_counts = (
    df_base.with_columns(pl.col("pub_date").dt.year().alias("pub_year"))
    .group_by("pub_year")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_year")
)

pub_year_counts

In [ ]:
pub_month_counts = (
    df_base
    .with_columns(
        pl.col("pub_date")
          .dt.truncate("1mo")
          .alias("pub_month")
    )
    .group_by("pub_month")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_month")
)

pub_month_counts

In [ ]:
top_keywords = (
    df_keywords
    .filter(pl.col("keyword").is_not_null())
    .group_by("keyword")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_keywords

In [ ]:
top_authors = (
    df_authors
    .filter(pl.col("author_full_name").is_not_null())
    .group_by("author_full_name")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_authors

#### Visualizing

In [ ]:
(
    alt.Chart(data_avail)
    .mark_bar()
    .encode(
        x=alt.X("data_available:O", title=""),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Data Availability")
)

In [ ]:
(
    alt.Chart(pub_year_counts)
    .mark_bar()
    .encode(
        x=alt.X("pub_year:O", title="Publication Year"),
        y=alt.Y("num_articles:Q", title="Number of Articles"),
    )
    .properties(title="Publications per Year")
)

In [ ]:
(
    alt.Chart(pub_month_counts)
    .mark_line(point=True)
    .encode(
        x=alt.X("pub_month:T", title="Publication Date"),
        y=alt.Y("num_articles:Q", title="Number of Articles")
    )
    .properties(title="Publications Over Time")
)

In [ ]:
(
    alt.Chart(top_keywords)
    .mark_bar()
    .encode(
        x=alt.X(
            "keyword:O",
            title="Keyword",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Count"),
    )
    .properties(title="Top 15 Keywords")
)

In [ ]:
(
    alt.Chart(top_authors)
    .mark_bar()
    .encode(
        x=alt.X(
            "author_full_name:O",
            title="Author",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Top 15 Authors")
)